In [ ]:
import pandas as pd

file_name = "nodes_anon_sample.csv"
nodes = pd.read_csv(file_name)

node_dict = {}           #Create a dictionary to store the (id,connection) pairs

for i in range(len(nodes)):             #Set every value to 0
    node_id = nodes.loc[i, 'id']
    node_dict[node_id] = 0

file_name = "edges_anon_sample.csv"
edges = pd.read_csv(file_name)

for i in range(len(edges)):              #increment their values in dictionary every time they are the source or target
    source = edges.loc[i, 'source']
    target = edges.loc[i, 'target']
    if source in node_dict:
        node_dict[source] += 1
    if target in node_dict:
        node_dict[target] += 1

sorted_nodes = sorted(node_dict.items(), key =lambda x: x[1], reverse=True)   #Sort the dictionary by their connections. Reverse the list(we want largest to smallest)

option = -1

while option != 0:
  print("1.Search a node")
  print("2.Print nodes with most connections")
  print("3.Print nodes with most outgoing connections")
  print("4.Print nodes with most incoming connections")
  print("0.Exit\n")
  option = int(input("Select an option: "))

  if option == 1:
    ID = input("Enter a node ID to search: ")
    print("")
    found = False
    for i in range(len(nodes)):
      if nodes.loc[i, 'id'] == ID:
        found = True
        print("Node ID:",nodes.loc[i,'id'])
        print("Connections:",node_dict[ID])
        print("Location:",nodes.loc[i, 'location'])
        print("Products:",nodes.loc[i, 'products'],"\n")
        break
    if found == False:
      print("Node not found")


  elif option == 2:
    inp = int(input("How many nodes do you want to include: "))
    print("")
    print("Top",inp,"nodes with most connections (incoming and outgoing): ")

    for i in range(inp):
      print(sorted_nodes[i][0],":" ,node_dict[sorted_nodes[i][0]])     #get the first inp elements of sorted list
    print("")

  elif option == 3:
    source_dict = {}
    for source in edges['source']:
        if source in source_dict:
            source_dict[source] += 1
        else:
            source_dict[source] = 1

    sorted_source = sorted(source_dict.items(), key =lambda x: x[1], reverse=True)
    inp = int(input("How many nodes do you want to include: "))
    print("")
    print("Top",inp,"nodes with outgoing connections: ")
    if sorted_source:
      for i in range(inp):
        print(sorted_source[i][0],":" ,source_dict[sorted_source[i][0]])
    print("")

  elif option == 4:
    target_dict = {}
    for target in edges['target']:
        if target in target_dict:
            target_dict[target] += 1
        else:
            target_dict[target] = 1

    sorted_target = sorted(target_dict.items(), key = lambda x: x[1], reverse=True)
    inp = int(input("How many nodes do you want to include: "))
    print("")
    print("Top",inp,"nodes with incoming connections: ")
    if sorted_target:
      for i in range(inp):
        print(sorted_target[i][0],":" ,target_dict[sorted_target[i][0]])
    print("")

  elif option == 0:
    print("Goodbye!")

#Possible options to add:
#2) Fewest Connections (very easy to implement)
#3) Neighbor search option
#4) Filter by country(or some other data)
#5)
#6)
#7)
#8)

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from math import sqrt

# 1) Veriyi yükle
path = "edges.csv"  # kendi yolunu ver
edges = pd.read_csv(path).rename(columns=str.lower)  # 'source', 'target'

# 2) Yönlü grafiği kur
G = nx.from_pandas_edgelist(edges, source="source", target="target", create_using=nx.DiGraph())

# 3) Ölçek için derece hesapları
in_deg  = dict(G.in_degree())
out_deg = dict(G.out_degree())
tot_deg = {n: in_deg.get(n,0) + out_deg.get(n,0) for n in G.nodes()}

def scale_sizes(d, min_size=40, max_size=800):
    vals = list(d.values())
    if not vals: return []
    vmin, vmax = min(vals), max(vals)
    if vmax == vmin: return [max_size for _ in vals]
    return [min_size + (max_size-min_size)*(v-vmin)/(vmax-vmin) for v in vals]

node_sizes = scale_sizes(tot_deg, min_size=30, max_size=600)

# 4) Yerleşim (reprodüksiyon için seed)
k_val = 1 / sqrt(max(len(G), 1))
pos = nx.spring_layout(G, k=k_val, seed=42)

# 5) Tüm ağ görseli (ok ucu kapalı, etiket yok -> performans/okunabilirlik)
plt.figure(figsize=(14,10))
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, linewidths=0.5)
nx.draw_networkx_edges(G, pos, width=0.3, alpha=0.25)  # arrows=False default
plt.axis("off")
plt.title("Supply Chain Network (Directed structure; edges drawn without arrowheads for performance)")
plt.tight_layout()
plt.savefig("supply_chain_network_full.png", dpi=250, bbox_inches="tight")
plt.close()

# 6) Odak alt ağ (Top-K hub’lar) — etiketli ve ok uçlu
K = 50  # ihtiyaca göre değiştir
top_nodes = [n for n,_ in sorted(tot_deg.items(), key=lambda x: x[1], reverse=True)[:K]]
H = G.subgraph(top_nodes).copy()

H_sizes = scale_sizes({n: tot_deg[n] for n in H.nodes()}, min_size=200, max_size=1500)
pos_H = nx.spring_layout(H, k=1/sqrt(max(len(H),1)), seed=7)

plt.figure(figsize=(14,10))
nx.draw_networkx_nodes(H, pos_H, node_size=H_sizes, linewidths=1)
nx.draw_networkx_edges(H, pos_H, arrows=True, width=1, alpha=0.6, arrowsize=12)
nx.draw_networkx_labels(H, pos_H, font_size=9)
plt.axis("off")
plt.title(f"Top-{K} Hubs (by Total Degree) — Labeled View")
plt.tight_layout()
plt.savefig(f"supply_chain_network_top{K}.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# save as compute_directed_loss.py and run: python compute_directed_loss.py
import re, networkx as nx, pandas as pd

edges_text = """
(n0,n4),(n0,n11),(n0,n18),(n0,n27),(n1,n3),(n1,n7),(n1,n14),(n1,n21),(n2,n6),(n2,n9),
(n2,n19),(n2,n25),(n3,n5),(n3,n12),(n3,n17),(n3,n30),(n4,n8),(n4,n15),(n4,n22),(n4,n31),
(n5,n10),(n5,n18),(n5,n27),(n5,n33),(n6,n2),(n6,n13),(n6,n20),(n6,n35),(n7,n4),(n7,n16),
(n7,n24),(n7,n29),(n8,n12),(n8,n19),(n8,n21),(n8,n39),(n9,n3),(n9,n14),(n9,n22),(n9,n41),
(n10,n10),(n10,n18),(n10,n26),(n10,n32),(n11,n7),(n11,n13),(n11,n28),(n11,n35),(n12,n8),(n12,n20),
(n12,n23),(n12,n40),(n13,n9),(n13,n17),(n13,n25),(n13,n37),(n14,n6),(n14,n21),(n14,n33),(n14,n42),
(n15,n12),(n15,n23),(n15,n27),(n15,n36),(n16,n11),(n16,n19),(n16,n28),(n16,n41),(n17,n7),(n17,n24),
(n17,n31),(n17,n38),(n18,n14),(n18,n22),(n18,n29),(n18,n43),(n19,n15),(n19,n26),(n19,n34),(n19,n46),
(n20,n13),(n20,n25),(n20,n37),(n20,n48),(n21,n9),(n21,n28),(n21,n32),(n21,n44),(n22,n16),(n22,n30),
(n22,n35),(n22,n47),(n23,n18),(n23,n27),(n23,n39),(n23,n45),(n24,n12),(n24,n29),(n24,n41),(n24,n49),
(n25,n25),(n25,n20),(n25,n33),(n25,n42),(n26,n14),(n26,n30),(n26,n38),(n26,n50),(n27,n17),(n27,n32),
(n27,n36),(n27,n45),(n28,n21),(n28,n34),(n28,n40),(n28,n47),(n29,n24),(n29,n31),(n29,n43),(n29,n50),
(n30,n18),(n30,n33),(n30,n37),(n30,n46),(n31,n22),(n31,n38),(n31,n44),(n31,n48),(n32,n23),(n32,n35),
(n32,n41),(n32,n49),(n33,n26),(n33,n39),(n33,n45),(n33,n50),(n34,n29),(n34,n30),(n34,n46),(n34,n47),
(n35,n24),(n35,n36),(n35,n43),(n35,n48),(n36,n31),(n36,n40),(n36,n44),(n36,n49),(n37,n37),(n37,n32),
(n37,n42),(n37,n50),(n38,n33),(n38,n41),(n38,n47),(n38,n48),(n39,n28),(n39,n36),(n39,n45),(n39,n50),
(n40,n30),(n40,n34),(n40,n46),(n40,n49),(n41,n22),(n41,n35),(n41,n43),(n41,n48),(n42,n24),(n42,n37),
(n42,n44),(n42,n50),(n43,n21),(n43,n38),(n43,n47),(n43,n49),(n44,n33),(n44,n39),(n44,n45),(n44,n48),
(n45,n36),(n45,n41),(n45,n46),(n45,n50),(n46,n29),(n46,n40),(n46,n47),(n46,n49),(n47,n31),(n47,n43),
(n47,n48),(n47,n50),(n48,n35),(n48,n44),(n48,n46),(n48,n49),(n49,n28),(n49,n37),(n49,n42),(n49,n50),
(n50,n50)
"""

pairs = re.findall(r'\(n\d+\,n\d+\)', edges_text)
edges = [tuple(p.strip('()').split(',')) for p in pairs]

# Build directed graph
G = nx.DiGraph()
G.add_nodes_from([f"n{i}" for i in range(51)])
for s,t in edges:
    G.add_edge(s,t)

nodes = sorted(G.nodes())

# Original reachability sets (nodes reachable FROM u)
orig_reach = {u: set(nx.descendants(G, u)) for u in nodes}

# Build loss matrix: rows = deleted node, cols = affected node
loss_mat = pd.DataFrame(0, index=nodes, columns=nodes)

for d in nodes:
    G2 = G.copy()
    if d in G2:
        G2.remove_node(d)
    for u in nodes:
        before = set(orig_reach[u])
        # after: if u==d -> empty. else compute descendants in G2
        if u == d:
            after = set()
        else:
            if u not in G2.nodes():
                after = set()
            else:
                after = set(nx.descendants(G2, u))
        lost = before - after
        loss_mat.at[d, u] = len(lost)

# Save CSV
outpath = "directed_reachability_loss_matrix.csv"
loss_mat.index.name = "deleted_node"
loss_mat.to_csv(outpath)
print("Saved directed reachability-loss matrix to:", outpath)

# Top 10 deletions by total disruption
totals = loss_mat.sum(axis=1).sort_values(ascending=False)
print("\nTop 10 deletions by total directed reachability loss:")
print(totals.head(10))
# ==========================================================
#  EXTRA CRITICALITY TESTS: EDGE COUNT + BETWEENNESS
# ==========================================================

print()
print()
print()
print()

print("\n=== Computing Edge Test (Most Connections) ===")

# compute degree counts
edge_counts = {n: G.in_degree(n) + G.out_degree(n) for n in nodes}

# find maximum
max_edges = max(edge_counts.values())
top_connected_nodes = [n for n, deg in edge_counts.items() if deg == max_edges]

print(f"Highest connection count: {max_edges}")
print("Nodes with most connections:", top_connected_nodes)

# keep dataframe for later combined ranking
edge_test_df = pd.DataFrame.from_dict(edge_counts, orient='index', columns=["edge_count"])
edge_test_df.index.name = "node"




print("\n=== Computing Betweenness Centrality (Directed) ===")
bet_cent = nx.betweenness_centrality(G, normalized=True, endpoints=False)
bet_df = pd.DataFrame.from_dict(bet_cent, orient='index', columns=["betweenness"])
bet_df.index.name = "node"

bet_df = pd.DataFrame.from_dict(bet_cent, orient='index', columns=["betweenness"])
bet_df.index.name = "node"

# ============================
# SEPARATE RANKINGS (TOP 10 ONLY)
# ============================

print("\n=== Top 10 by Edge Count (Simple Degree) ===")
ranked_edges = sorted(edge_counts.items(), key=lambda x: x[1], reverse=True)
for n, deg in ranked_edges[:10]:
    print(f"{n}: {deg}")

print("\n=== Top 10 by Betweenness Centrality ===")
ranked_bc = sorted(bet_cent.items(), key=lambda x: x[1], reverse=True)
for n, bc in ranked_bc[:10]:
    print(f"{n}: {bc}")


# Combine results
results = edge_test_df.join(bet_df)

# Rank nodes (larger = more critical)
results["rank_edge"] = results["edge_count"].rank(ascending=False, method="dense")
results["rank_bc"] = results["betweenness"].rank(ascending=False, method="dense")

# Weighted combined ranking (optional):
# You can adjust weights here
results["combined_score"] = (results["rank_edge"] + results["rank_bc"]) / 2
results["combined_rank"] = results["combined_score"].rank(ascending=True, method="dense")

results_sorted = results.sort_values("combined_rank")
print("\n=== Top 10 Critical Nodes Based on Combined Ranking ===")
print(results_sorted.head(10))

# Save
results.to_csv("directed_edge_and_betweenness_scores.csv")
print("\nSaved node criticality scores to: directed_edge_and_betweenness_scores.csv")



# ============================
# MOST VULNERABLE NODES
# ============================

vulnerability_count = {}

for u in nodes:
     count = sum(1 for d in nodes if d != u and loss_mat.at[d, u] > 0)
     vulnerability_count[u] = count

# convert to DataFrame for easy sorting
vuln_df = pd.DataFrame.from_dict(vulnerability_count, orient='index', columns=["affected_by"])
vuln_df.index.name = "node"

# sort descending to get top 10 most vulnerable
vuln_df_sorted = vuln_df.sort_values("affected_by", ascending=False)

print("\n=== Top 10 Most Vulnerable Nodes ===")
print(vuln_df_sorted.head(10))


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import re

# Parse the edges from the given text
edges_text = """
(n0,n4),(n0,n11),(n0,n18),(n0,n27),(n1,n3),(n1,n7),(n1,n14),(n1,n21),(n2,n6),(n2,n9),
(n2,n19),(n2,n25),(n3,n5),(n3,n12),(n3,n17),(n3,n30),(n4,n8),(n4,n15),(n4,n22),(n4,n31),
(n5,n10),(n5,n18),(n5,n27),(n5,n33),(n6,n2),(n6,n13),(n6,n20),(n6,n35),(n7,n4),(n7,n16),
(n7,n24),(n7,n29),(n8,n12),(n8,n19),(n8,n21),(n8,n39),(n9,n3),(n9,n14),(n9,n22),(n9,n41),
(n10,n10),(n10,n18),(n10,n26),(n10,n32),(n11,n7),(n11,n13),(n11,n28),(n11,n35),(n12,n8),(n12,n20),
(n12,n23),(n12,n40),(n13,n9),(n13,n17),(n13,n25),(n13,n37),(n14,n6),(n14,n21),(n14,n33),(n14,n42),
(n15,n12),(n15,n23),(n15,n27),(n15,n36),(n16,n11),(n16,n19),(n16,n28),(n16,n41),(n17,n7),(n17,n24),
(n17,n31),(n17,n38),(n18,n14),(n18,n22),(n18,n29),(n18,n43),(n19,n15),(n19,n26),(n19,n34),(n19,n46),
(n20,n13),(n20,n25),(n20,n37),(n20,n48),(n21,n9),(n21,n28),(n21,n32),(n21,n44),(n22,n16),(n22,n30),
(n22,n35),(n22,n47),(n23,n18),(n23,n27),(n23,n39),(n23,n45),(n24,n12),(n24,n29),(n24,n41),(n24,n49),
(n25,n25),(n25,n20),(n25,n33),(n25,n42),(n26,n14),(n26,n30),(n26,n38),(n26,n50),(n27,n17),(n27,n32),
(n27,n36),(n27,n45),(n28,n21),(n28,n34),(n28,n40),(n28,n47),(n29,n24),(n29,n31),(n29,n43),(n29,n50),
(n30,n18),(n30,n33),(n30,n37),(n30,n46),(n31,n22),(n31,n38),(n31,n44),(n31,n48),(n32,n23),(n32,n35),
(n32,n41),(n32,n49),(n33,n26),(n33,n39),(n33,n45),(n33,n50),(n34,n29),(n34,n30),(n34,n46),(n34,n47),
(n35,n24),(n35,n36),(n35,n43),(n35,n48),(n36,n31),(n36,n40),(n36,n44),(n36,n49),(n37,n37),(n37,n32),
(n37,n42),(n37,n50),(n38,n33),(n38,n41),(n38,n47),(n38,n48),(n39,n28),(n39,n36),(n39,n45),(n39,n50),
(n40,n30),(n40,n34),(n40,n46),(n40,n49),(n41,n22),(n41,n35),(n41,n43),(n41,n48),(n42,n24),(n42,n37),
(n42,n44),(n42,n50),(n43,n21),(n43,n38),(n43,n47),(n43,n49),(n44,n33),(n44,n39),(n44,n45),(n44,n48),
(n45,n36),(n45,n41),(n45,n46),(n45,n50),(n46,n29),(n46,n40),(n46,n47),(n46,n49),(n47,n31),(n47,n43),
(n47,n48),(n47,n50),(n48,n35),(n48,n44),(n48,n46),(n48,n49),(n49,n28),(n49,n37),(n49,n42),(n49,n50),
(n50,n50)
"""

pairs = re.findall(r'\(n(\d+),n(\d+)\)', edges_text)
edges = [(f"n{a}", f"n{b}") for a, b in pairs]

# Build graph
G = nx.Graph()
G.add_edges_from(edges)

# Plot
plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_size=300, font_size=8)
plt.show()


In [ ]:
import pandas as pd
import networkx as nx
import graphviz as gv

# ==========================================
# 1. VERİ HAZIRLIĞI
# ==========================================
data = {
    'sourceStage': [
        'Manuf_0001', 'Manuf_0002', 'Manuf_0003', 'Manuf_0004', 'Manuf_0004',
        'Part_0001', 'Part_0002', 'Part_0002', 'Part_0003', 'Part_0003',
        'Part_0004', 'Part_0005', 'Part_0005', 'Trans_0001', 'Trans_0002',
        'Trans_0003', 'Trans_0004', 'Trans_0004'
    ],
    'destinationStage': [
        'Trans_0001', 'Trans_0004', 'Dist_0002', 'Dist_0003', 'Dist_0004',
        'Manuf_0001', 'Manuf_0001', 'Manuf_0002', 'Manuf_0001', 'Manuf_0002',
        'Manuf_0002', 'Trans_0002', 'Trans_0003', 'Manuf_0003', 'Manuf_0003',
        'Manuf_0004', 'Dist_0001', 'Manuf_0004'
    ]
}
df = pd.DataFrame(data)
G = nx.from_pandas_edgelist(df, 'sourceStage', 'destinationStage', create_using=nx.DiGraph())
all_nodes = sorted(list(G.nodes()))

# Her düğümün girdilerini (predecessors) önceden kaydedelim
# Çünkü simülasyonda "Bana gelen kaynaklar hala hayatta mı?" diye soracağız.
node_predecessors = {n: set(G.predecessors(n)) for n in all_nodes}

# ==========================================
# 2. ZİNCİRLEME ÇÖKÜŞ SİMÜLASYONU (CASCADING FAILURE)
# ==========================================

cascade_scores = {}

for target_node in all_nodes:
    # 1. Adım: Düğümü siliyoruz (Bu düğüm "fail" oldu)
    failed_nodes = {target_node}

    # 2. Adım: Dalga dalga yayılma (Propagation)
    while True:
        newly_failed = set()

        # Henüz bozulmamış (sağlam) düğümleri kontrol et
        remaining_nodes = [n for n in all_nodes if n not in failed_nodes]

        for node in remaining_nodes:
            # Bu düğümün çalışması için gerekli girdiler (predecessors)
            required_inputs = node_predecessors[node]

            # Eğer bu düğümün HİÇ girdisi yoksa (örn: Part'lar), ve kendisi silinmediyse çalışır.
            if not required_inputs:
                continue

            # KURAL: Eğer gerekli girdilerden HERHANGİ BİRİ (ANY) bozulmuşsa,
            # bu düğüm de çalışmayı durdurur (Montaj yapılamaz).
            # intersection kümesi boş değilse, bozuk bir girdi var demektir.
            if not required_inputs.isdisjoint(failed_nodes):
                newly_failed.add(node)

        # Eğer bu turda yeni bozulan bir şey yoksa döngü biter
        if not newly_failed:
            break

        # Yeni bozulanları "failed" listesine ekle ve döngüye devam et
        failed_nodes.update(newly_failed)

    # 3. Adım: Skor Hesapla
    # Skor = Toplam bozulan sayısı - 1 (Kendisini saymıyoruz, etkiyi ölçüyoruz)
    cascade_scores[target_node] = len(failed_nodes) - 1

# Tabloyu oluştur
df_scores = pd.DataFrame(list(cascade_scores.items()), columns=['Node', 'Cascade_Impact'])
df_scores = df_scores.sort_values(by='Cascade_Impact', ascending=False).reset_index(drop=True)

print("=== YENİ SKORLAR (ZİNCİRLEME ETKİ) ===")
print("Not: Artık Supplier'lar (Part) en yüksek etkiye sahip olmalı.")
print(df_scores)

# ==========================================
# 3. GÖRSELLEŞTİRME
# ==========================================

# Renk Ayarları
COLOR_MAP = { "Part": "#F0F0F0", "Manuf": "#ADD8E6", "Trans": "#98FB98", "Dist": "#FFB6C1" }
def get_fill_color(n):
    for k, v in COLOR_MAP.items():
        if n.startswith(k): return v
    return "white"

max_impact = df_scores['Cascade_Impact'].max()

dot = gv.Digraph(comment='Cascading Failure Map', format='png')
dot.attr(rankdir='LR', splines='true', nodesep='0.6')
dot.attr('node', fontname='Arial', fontsize='10', style='filled')

for node in all_nodes:
    score = cascade_scores[node]
    fill_col = get_fill_color(node)

    # Görselleştirme Mantığı:
    # Skor yüksekse -> Kırmızı ve Kalın
    # Skor düşükse -> Siyah ve İnce

    if score > 0:
        # Renk skalası: Skor ne kadar büyükse o kadar koyu kırmızı çerçeve
        # Basitçe: 0'dan büyükse kırmızı yapıyoruz.
        border_color = 'red'
        # Kalınlık: 1 ile 6 arasında oranla
        width_val = 1.0 + (score / max_impact) * 5.0
        label = f"{node}\n(Impact: {score})"
        font_col = 'black'
        # Eğer çok kritikse (üst %20) yazı da kırmızı olsun
        if score > max_impact * 0.8:
            font_col = 'red'
            font_weight = 'bold'
        else:
            font_weight = 'normal'

    else:
        border_color = 'black'
        width_val = 1.0
        label = f"{node}\n(Impact: 0)"
        font_col = 'gray'
        font_weight = 'normal'

    dot.node(node, label,
             shape='box',
             fillcolor=fill_col,
             color=border_color,
             penwidth=str(width_val),
             fontcolor=font_col)

# Kenarlar
for u, v in G.edges():
    dot.edge(u, v, color='#555555', arrowhead='vee')

print("\nGörsel oluşturuluyor...")
display(dot)

In [ ]:
import pandas as pd
import networkx as nx
import graphviz as gv
import os

# ==========================================
# 1. AYARLAR VE CSV YÜKLEME
# ==========================================

# Dosya adınızı buraya yazın
csv_dosya_adi = 'tedarik_zinciri.csv'

# Eğer elinizde dosya yoksa, test etmeniz için burada geçici bir tane oluşturuyorum:
# (Kendi dosyanızı kullanacağınız zaman bu if bloğunu silebilirsiniz)
if not os.path.exists(csv_dosya_adi):
    print(f"{csv_dosya_adi} bulunamadı, örnek veri ile oluşturuluyor...")
    sample_data = """sourceStage,destinationStage
Manuf_0001,Trans_0001
Manuf_0002,Trans_0004
Manuf_0003,Dist_0002
Manuf_0004,Dist_0003
Manuf_0004,Dist_0004
Part_0001,Manuf_0001
Part_0002,Manuf_0001
Part_0002,Manuf_0002
Part_0003,Manuf_0001
Part_0003,Manuf_0002
Part_0004,Manuf_0002
Part_0005,Trans_0002
Part_0005,Trans_0003
Trans_0001,Manuf_0003
Trans_0002,Manuf_0003
Trans_0003,Manuf_0004
Trans_0004,Dist_0001
Trans_0004,Manuf_0004
"""
    with open(csv_dosya_adi, "w") as f:
        f.write(sample_data)

# --- CSV OKUMA İŞLEMİ ---
try:
    df = pd.read_csv(csv_dosya_adi)

    # Sütun isimlerini kontrol edelim, eğer farklıysa burayı düzeltin
    kaynak_sutun = 'sourceStage'
    hedef_sutun = 'destinationStage'

    # Veri Temizliği (Strip): " Part_001 " gibi boşluk hatalarını siler
    df[kaynak_sutun] = df[kaynak_sutun].astype(str).str.strip()
    df[hedef_sutun] = df[hedef_sutun].astype(str).str.strip()

    print(f"CSV Başarıyla Yüklendi. Toplam {len(df)} bağlantı var.")

except Exception as e:
    print(f"Hata: CSV dosyası okunamadı. Lütfen dosya adını kontrol edin.\nHata Detayı: {e}")
    exit() # Hata varsa durdur

# ==========================================
# 2. AĞI OLUŞTURMA VE ANALİZ
# ==========================================

G = nx.from_pandas_edgelist(df, kaynak_sutun, hedef_sutun, create_using=nx.DiGraph())
all_nodes = sorted(list(G.nodes()))

# Her düğümün ihtiyaç duyduğu girdileri (predecessors) kaydet
node_predecessors = {n: set(G.predecessors(n)) for n in all_nodes}

# --- CASCADING FAILURE (Zincirleme Çöküş) SİMÜLASYONU ---
cascade_scores = {}

for target_node in all_nodes:
    # 1. Hedef düğümü boz
    failed_nodes = {target_node}

    # 2. Etkiyi yay (Propagate)
    while True:
        newly_failed = set()
        remaining_nodes = [n for n in all_nodes if n not in failed_nodes]

        for node in remaining_nodes:
            required_inputs = node_predecessors[node]

            # Eğer girdisi yoksa (Supplier) ve kendisi bozulmadıysa çalışır.
            if not required_inputs:
                continue

            # KURAL: Girdilerden HERHANGİ BİRİ bozuksa, bu düğüm de durur.
            if not required_inputs.isdisjoint(failed_nodes):
                newly_failed.add(node)

        if not newly_failed:
            break
        failed_nodes.update(newly_failed)

    # Kendisi hariç kaç yeri etkiledi?
    cascade_scores[target_node] = len(failed_nodes) - 1

# Sonuçları DataFrame yap
df_scores = pd.DataFrame(list(cascade_scores.items()), columns=['Node', 'Cascade_Impact'])
df_scores = df_scores.sort_values(by='Cascade_Impact', ascending=False).reset_index(drop=True)

print("\n=== EN KRİTİK NOKTALAR (Etki Puanına Göre) ===")
print(df_scores.head(10))

# ==========================================
# 3. GÖRSELLEŞTİRME (GRAPHVIZ)
# ==========================================

# Renk Tanımları
COLOR_MAP = { "Part": "#F0F0F0", "Manuf": "#ADD8E6", "Trans": "#98FB98", "Dist": "#FFB6C1" }
def get_fill_color(n):
    for k, v in COLOR_MAP.items():
        if n.startswith(k): return v
    return "white" # Varsayılan

max_impact = df_scores['Cascade_Impact'].max() if not df_scores.empty else 1

dot = gv.Digraph(comment='Supply Chain From CSV', format='png')
dot.attr(rankdir='LR', splines='true', nodesep='0.6', ranksep='0.8')
dot.attr('node', fontname='Arial', fontsize='10', style='filled')

for node in all_nodes:
    score = cascade_scores.get(node, 0)
    fill_col = get_fill_color(node)

    # Stil Mantığı
    if score > 0:
        border_color = 'red'
        # Çerçeve kalınlığı puana göre artar (Min 1, Max 6)
        pen_width = str(1.0 + (score / max_impact) * 5.0)
        label = f"{node}\n(Etki: {score})"

        # Çok yüksek skorlarda yazı rengini de kırmızı yap
        if score > max_impact * 0.7:
            font_col = 'red'
            font_weight = 'bold'
        else:
            font_col = 'black'
            font_weight = 'normal'
    else:
        border_color = 'black'
        pen_width = '1.0'
        label = f"{node}\n(Etki: 0)"
        font_col = 'gray'
        font_weight = 'normal'

    dot.node(node, label,
             shape='box',
             fillcolor=fill_col,
             color=border_color,
             penwidth=pen_width,
             fontcolor=font_col,
             fontweight='bold' if font_weight == 'bold' else 'normal')

# Kenarları Çiz
for u, v in G.edges():
    dot.edge(u, v, color='#555555', arrowhead='vee')

print("\nGörsel oluşturuluyor...")
display(dot) # Jupyter için
# dot.render('supply_chain_analysis', view=True) # Dosya olarak kaydetmek için

# LAST VERSION-MMM


In [ ]:
#Echelon Format(Acyclic)
import pandas as pd
import networkx as nx
import numpy as np

# Dosyaları yükleyelim
nodes_df = pd.read_csv('Pure-Nodes-Info.csv')
edges_df = pd.read_csv('PURE-Edges.csv')

# --- STEP 1: SEMANTIC CLASSIFICATION  ---
def categorize_stage_semantic(product_desc):
    if pd.isna(product_desc):
        return "Unknown"

    desc = product_desc.lower()
    # Keyword listS (Supply Chain Logic)
    raw_materials = ['coal', 'ore', 'acid', 'wood', 'limestone', 'uranium', 'sodium',
                     'raw', 'copper', 'iron', 'manganese', 'steel', 'alumina', 'slime', 'pyrite']
    manufacturing = ['part of vehicle', 'assembly', 'engine', 'casting', 'transmission',
                     'semiconductor', 'diode', 'transistor', 'led', 'device', 'unit', 'brake', 'system', 'sensor']
    retail_services = ['household', 'food', 'beverage', 'software', 'insurance', 'banking',
                       'finance', 'real estate', 'apparel', 'textile', 'publishing', 'broadcasting']

    # Order Priority: First Raw Material
    if any(x in desc for x in raw_materials):
        return "Part"
    elif any(x in desc for x in manufacturing):
        return "Manuf"
    elif any(x in desc for x in retail_services):
        return "Retail"
    else:
        return "Dist"

# First Classification
nodes_df['initial_stage'] = nodes_df['products'].apply(categorize_stage_semantic)

# --- STEP 2: TOPOLOGICAL REFINEMENT---
G = nx.DiGraph()
for _, row in nodes_df.iterrows():
    G.add_node(row['id'], label=row['initial_stage'], desc=row['products'])
valid_edges = []
for _, row in edges_df.iterrows():
    if row['source'] in G.nodes and row['target'] in G.nodes:
        G.add_edge(row['source'], row['target'])
        valid_edges.append(row)

# Degree Calculations
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())

# Sınıflandırmayı topolojiye göre güncelleme
final_stages = {}
for node in G.nodes:
    stage = G.nodes[node]['label']

    # Kural 1: Eğer bir node'un girdisi yoksa (in-degree 0) ve 'Unknown' veya 'Dist' ise -> PART olmalı
    if in_degrees[node] == 0:
        final_stages[node] = "Part"

    # Kural 2: Eğer bir node'un çıktısı yoksa (out-degree 0) -> RETAIL olmalı (Üretim olsa bile müşteriye gitmiyorsa son duraktır)
    elif out_degrees[node] == 0:
        final_stages[node] = "Retail"

    # Kural 3: Diğer durumlarda semantik analizi koru, eğer semantic 'Unknown' ise ve arada kalmışsa 'Dist' yap
    else:
        if stage == "Unknown":
            final_stages[node] = "Dist"
        else:
            final_stages[node] = stage

# Node özelliklerini güncelle
nx.set_node_attributes(G, final_stages, 'stage')

# --- ADIM 3: CYCLE BREAKING (Döngü Kırma) ---
# Hiyerarşi Skoru: Part(1) -> Manuf(2) -> Dist(3) -> Retail(4)
stage_hierarchy = {"Part": 1, "Manuf": 2, "Dist": 3, "Retail": 4}

removed_edges_hierarchy = []
edges_to_remove = []

# Yöntem A: Hiyerarşik Temizlik (Örn: Retail -> Part akışını sil)
for u, v in G.edges():
    u_rank = stage_hierarchy[final_stages[u]]
    v_rank = stage_hierarchy[final_stages[v]]

    # Eğer akış ters yöndeyse (Yüksek rank'ten düşük rank'e) -> Kesinlikle sil
    if u_rank > v_rank:
        edges_to_remove.append((u, v))
    # Not: u_rank == v_rank durumu (Örn: Manuf -> Manuf) kabul edilebilir, şimdilik tutuyoruz.

G.remove_edges_from(edges_to_remove)
removed_edges_count_hierarchy = len(edges_to_remove)

# Yöntem B: Kalan Döngüleri Temizleme (DFS ile)
# Hiyerarşik temizlikten sonra bile kendi içinde (Manuf A -> Manuf B -> Manuf A) döngüler olabilir.
initial_cycles = list(nx.simple_cycles(G))
cycle_breaking_edges = []

while True:
    try:
        cycle = nx.find_cycle(G, orientation='original')
        # Döngüyü kırmak için cycle içindeki son kenarı (back edge) seçip siliyoruz
        # Daha akıllıca olması için: Döngüdeki kenarlardan "Manuf->Manuf" gibi olanlardan birini seçiyoruz.
        u, v = cycle[-1][:2] # Son kenarı al
        G.remove_edge(u, v)
        cycle_breaking_edges.append((u, v))
    except nx.NetworkXNoCycle:
        break

final_edge_count = G.number_of_edges()
final_node_count = G.number_of_nodes()

# İstatistikleri Raporlama
stage_counts = pd.Series(final_stages).value_counts()

In [ ]:
# --- RAPORLAMA VE KONTROL BLOĞU ---

print(f"Başlangıç Edge Sayısı: {len(edges_df)}")
print(f"Bitiş Edge Sayısı: {G.number_of_edges()}")
print(f"Silinen Edge Sayısı: {len(edges_df) - G.number_of_edges()}")

# Cycle Kontrolü
try:
    cycle = nx.find_cycle(G)
    print("UYARI: Hala cycle var! ->", cycle)
except nx.NetworkXNoCycle:
    print("BAŞARILI: Graph tamamen 'Acyclic' (Döngüsüz) hale geldi. Hiçbir loop kalmadı.")

# Sınıflandırma İstatistiği
print("\n--- Node Sınıflandırma Dağılımı ---")
stage_counts = pd.Series(nx.get_node_attributes(G, 'stage')).value_counts()
print(stage_counts)

# Örnek: Eskiden Unknown olup şimdi atananlar (İlk 5 örnek)
print("\n--- 'Unknown' Düzeltme Örnekleri ---")
count = 0
for node in nodes_df[nodes_df['initial_stage'] == 'Unknown']['id']:
    if count < 5:
        print(f"Node: {node}, Ürün: {G.nodes[node]['desc']}, Yeni Atama: {G.nodes[node]['stage']}")
        count += 1

In [ ]:
# --- ÇIKTI ÜRETME VE İNDİRME ---

# 1. Yeni Node Dosyası (Stage etiketi eklenmiş)
node_data = []
for n, data in G.nodes(data=True):
    node_data.append({
        'id': n,
        'stage_classification': data['stage'], # Part, Manuf, Dist, Retail
        'product_description': data.get('desc', '')
    })
clean_nodes_df = pd.DataFrame(node_data)
clean_nodes_df.to_csv('Cleaned_Nodes_Stage.csv', index=False)

# 2. Yeni Edge Dosyası (Cycle yaratanlar silinmiş)
edge_data = []
for u, v in G.edges():
    edge_data.append({'source': u, 'target': v})
clean_edges_df = pd.DataFrame(edge_data)
clean_edges_df.to_csv('Cleaned_Edges_Acyclic.csv', index=False)

print("\nDosyalar oluşturuldu: 'Cleaned_Nodes_Stage.csv' ve 'Cleaned_Edges_Acyclic.csv'")
# Eğer Colab kullanıyorsan dosyaları panele düşürecektir, oradan indirebilirsin.

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

# 1. Temizlenmiş Veriyi Yükle (Önceki adımda oluşturduklarımızı kullanıyoruz)
# Not: Eğer dosyalar panelde yoksa önceki kodun çıktısını tekrar çalıştırabilirsin.
nodes_df = pd.read_csv('Cleaned_Nodes_Stage.csv')
edges_df = pd.read_csv('Cleaned_Edges_Acyclic.csv')

G = nx.DiGraph()
for _, row in nodes_df.iterrows():
    G.add_node(row['id'], stage=row['stage_classification'], desc=row['product_description'])
for _, row in edges_df.iterrows():
    G.add_edge(row['source'], row['target'])

print("Graph yüklendi. Analiz başlıyor...")

# --- METRİK 1: Downstream Impact (Reverse Reachability) ---
# Bu node'dan aşağıya doğru gidildiğinde kaç farklı node'a ulaşılabiliyor?
downstream_impact = {}
for node in G.nodes():
    # descendants: bu node'dan gidilebilen tüm node'ların kümesi
    downstream_impact[node] = len(nx.descendants(G, node))

# --- METRİK 2: Betweenness Centrality ---
# Node'un ağdaki "kısa yolların" yüzde kaçında köprü olduğu
betweenness = nx.betweenness_centrality(G)

# --- METRİK 3: Out-Degree ---
out_degree = dict(G.out_degree())

# --- SKORLAMA VE NORMALİZASYON ---
# Verileri DataFrame'e alalım
metrics_df = pd.DataFrame({
    'node_id': G.nodes(),
    'raw_downstream': [downstream_impact.get(n, 0) for n in G.nodes()],
    'raw_betweenness': [betweenness.get(n, 0) for n in G.nodes()],
    'raw_out_degree': [out_degree.get(n, 0) for n in G.nodes()]
})

# Verileri 0-1 arasına sıkıştır (MinMax Scaling) ki elmalarla armutlar toplanabilsin
scaler = MinMaxScaler()
metrics_df[['norm_downstream', 'norm_betweenness', 'norm_out_degree']] = scaler.fit_transform(
    metrics_df[['raw_downstream', 'raw_betweenness', 'raw_out_degree']]
)

# --- Weighted Score Calculations ---
w_downstream = 0.50
w_betweenness = 0.30
w_out_degree = 0.20

metrics_df['CRITICAL_SCORE'] = (
    metrics_df['norm_downstream'] * w_downstream +
    metrics_df['norm_betweenness'] * w_betweenness +
    metrics_df['norm_out_degree'] * w_out_degree
)

# Sorting the results
top_critical_nodes = metrics_df.sort_values(by='CRITICAL_SCORE', ascending=False).head(10)

# Adding details about the critical nodes
top_critical_nodes = top_critical_nodes.merge(nodes_df, left_on='node_id', right_on='id', how='left')

print("\n--- TOP 10 CRITICAL NODES ---")
print(top_critical_nodes[['id', 'product_description', 'stage_classification', 'CRITICAL_SCORE']])

In [ ]:
# --- GÖRSELLEŞTİRME: FOCUSED SUB-GRAPH ---

# 1. Çizilecek Alt Kümeyi Belirle (Top 10 Kritik + Onların Komşuları)
critical_ids = set(top_critical_nodes['id'])
neighbors = set()
for node in critical_ids:
    neighbors.update(G.predecessors(node)) # Girdiler
    neighbors.update(G.successors(node))   # Çıktılar

nodes_to_draw = critical_ids.union(neighbors)
subgraph = G.subgraph(nodes_to_draw)

# 2. Renk Kodlaması
color_map = []
node_sizes = []
for node in subgraph.nodes():
    if node in critical_ids:
        color_map.append('red') # Kritikler Kırmızı
        node_sizes.append(1000) # Ve Büyük
    else:
        # Stage'e göre renk
        stage = subgraph.nodes[node]['stage']
        if stage == 'Part': color_map.append('#1f77b4') # Mavi
        elif stage == 'Manuf': color_map.append('#ff7f0e') # Turuncu
        elif stage == 'Dist': color_map.append('#2ca02c') # Yeşil
        elif stage == 'Retail': color_map.append('#9467bd') # Mor
        node_sizes.append(300) # Normal Boyut

# 3. Layout (Düzen) - Hiyerarşik (Part -> Manuf -> Dist -> Retail)
pos = {}
for node in subgraph.nodes():
    stage = subgraph.nodes[node]['stage']
    # X ekseni: Stage sırası
    if stage == 'Part': x = 0
    elif stage == 'Manuf': x = 1
    elif stage == 'Dist': x = 2
    else: x = 3

    # Y ekseni: Rastgele dağıt ama üst üste binmesin
    import random
    y = random.random()
    pos[node] = (x, y)

# 4. Çizim
plt.figure(figsize=(15, 10))
nx.draw(subgraph, pos, node_color=color_map, with_labels=True,
        node_size=node_sizes, edge_color='gray', alpha=0.7, font_size=8, arrowsize=15)

# Legend (Lejant)
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Critical Node', markerfacecolor='red', markersize=15),
    Line2D([0], [0], marker='o', color='w', label='Part', markerfacecolor='#1f77b4', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Manuf', markerfacecolor='#ff7f0e', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Dist', markerfacecolor='#2ca02c', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Retail', markerfacecolor='#9467bd', markersize=10)
]
plt.legend(handles=legend_elements, loc='upper right')
plt.title("Supply Chain Critical Nodes & Neighbors (Echelon View)")
plt.show()

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm # İlerleme çubuğu için

# --- ADIM 1: TEMİZLENMİŞ VERİYİ YÜKLEME ---
print("Veriler yükleniyor ve Graph oluşturuluyor...")
nodes_df = pd.read_csv('Cleaned_Nodes_Stage.csv')
edges_df = pd.read_csv('Cleaned_Edges_Acyclic.csv')

# Yönlü Graph (DiGraph) oluşturma
G = nx.DiGraph()
for _, row in nodes_df.iterrows():
    G.add_node(row['id'], stage=row['stage_classification'], desc=row['product_description'])
for _, row in edges_df.iterrows():
    G.add_edge(row['source'], row['target'])

# 'Part' (Hammadde) Node'larını belirle (Akışın kaynağı bunlar)
part_nodes = [n for n, attr in G.nodes(data=True) if attr['stage'] == 'Part']
all_nodes = list(G.nodes())

print(f"Toplam Node: {len(all_nodes)}")
print(f"Kaynak (Part) Node Sayısı: {len(part_nodes)}")
print("-" * 30)

# --- ADIM 2: DISRUPTION SIMULATION (SİMÜLASYON) ---
# Mantık: Her node'u tek tek silip, geri kalanların 'Part'lara ulaşıp ulaşamadığına bakacağız.

disruption_data = [] # Matrisi oluşturacak veriyi burada tutacağız

print("Simülasyon Başladı: Her node tek tek 'devre dışı' bırakılıyor...")

# Tqdm ile ilerleme çubuğu gösterelim (Yaklaşık 1-2 dk sürebilir)
for node_to_remove in tqdm(all_nodes, desc="Simulating Failures"):

    # 1. Node'u geçici olarak sil (Sanal bir kopya üzerinde)
    # Hızlı yöntem: Node silmek yerine, reachability analizi yapacağız.
    # Node silindiğinde o node üzerinden geçen yollar iptal olur.

    H = G.copy()
    H.remove_node(node_to_remove)

    # 2. Kalan ağda kimler hala 'Part'lara bağlı?
    # Mevcut Part'lar (Silinen node bir Part ise onu listeden çıkar)
    active_parts = [p for p in part_nodes if p != node_to_remove]

    # H graph'ında 'active_parts' listesinden aşağıya doğru erişilebilen tüm node'ları bul.
    # Bu set, 'Güvende' olanlardır.
    reachable_nodes = set()
    for part in active_parts:
        # Bu part'tan gidilebilen her yer (kendisi dahil)
        reachable_nodes.add(part)
        reachable_nodes.update(nx.descendants(H, part))

    # 3. Kimler Aç Kaldı? (Starved / Isolated)
    # Graph'ta var olup da 'reachable' kümesinde olmayanlar
    # (Node_to_remove zaten yok, onu saymıyoruz)
    remaining_nodes = set(H.nodes())
    starved_nodes = remaining_nodes - reachable_nodes

    # 4. Veriyi Kaydet
    # Dictionary formatında: {Mağdur_Node: 1, Diğerleri: 0}
    # Bu yöntem memory dostudur. Sadece etkilenenleri 1 yaparız.
    row_result = {n: 1 for n in starved_nodes}
    row_result['DISRUPTED_NODE'] = node_to_remove # Satır ismi (Kim patladı?)

    disruption_data.append(row_result)

# --- ADIM 3: MATRİS OLUŞTURMA VE ANALİZ ---
print("\nMatris oluşturuluyor...")
# DataFrame'e çevir (Boş yerler 0 olacak, yani etkilenmedi)
disruption_matrix = pd.DataFrame(disruption_data).fillna(0)
disruption_matrix.set_index('DISRUPTED_NODE', inplace=True)

# 1. VULNERABILITY SCORE (Sütun Toplamı)
# "Ben kaç farklı senaryoda ürünsüz kalıyorum?"
vulnerability_scores = disruption_matrix.sum(axis=0).sort_values(ascending=False)

# 2. CRITICALITY SCORE (Satır Toplamı)
# "Ben patlarsam kaç kişi ürünsüz kalıyor?"
criticality_scores = disruption_matrix.sum(axis=1).sort_values(ascending=False)

# --- ADIM 4: RAPORLAMA VE ÇIKTI ---

# En Kırılgan 10 Node
print("\n" + "="*40)
print("TOP 10 VULNERABLE NODES (EN KIRILGANLAR)")
print("Tanım: Başkaları hata yaptığında en sık 'Isolated' kalanlar.")
print("="*40)

top_vulnerable = vulnerability_scores.head(10).index
for node in top_vulnerable:
    score = vulnerability_scores[node]
    desc = G.nodes[node].get('desc', 'Unknown')
    stage = G.nodes[node].get('stage', 'Unknown')
    # Tedarikçi sayısı (In-Degree) bilgisini de ekleyelim, yorumlamak kolay olsun
    in_deg = G.in_degree(node)
    print(f"Node: {node} | Score: {int(score)} | Stage: {stage} | In-Degree: {in_deg} | Ürün: {desc}")

# Dosyaları Kaydetme
print("\nDosyalar kaydediliyor...")
disruption_matrix.to_csv('Full_Disruption_Matrix.csv')
vulnerability_scores.to_csv('Vulnerability_Ranking.csv')
print("-> 'Full_Disruption_Matrix.csv' (Dikkat: Büyük dosya olabilir)")
print("-> 'Vulnerability_Ranking.csv' (Tüm liste)")

# --- ADIM 5: GÖRSELLEŞTİRME (Bar Chart) ---
plt.figure(figsize=(10, 6))
# İlk 15 Vulnerable Node
top_15 = vulnerability_scores.head(15)
sns.barplot(x=top_15.values, y=top_15.index, palette="Reds_r")
plt.xlabel('Vulnerability Score (Kaç farklı senaryoda izole kaldı?)')
plt.ylabel('Node ID')
plt.title('Supply Chain Vulnerability Analysis (Top 15)')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import pandas as pd
import networkx as nx

# --- ADIM 1: GÜVENLİ VERİ YÜKLEME ---
print("1. Veriler okunuyor...")
nodes_df = pd.read_csv('Cleaned_Nodes_Stage.csv')
edges_df = pd.read_csv('Cleaned_Edges_Acyclic.csv')

# Sütun isimlerini garantiye alalım (Hata riskini sıfırlıyoruz)
# Eğer 'stage_classification' sütunu varsa onu kullan, yoksa 'stage' sütununa bak
stage_col = 'stage_classification' if 'stage_classification' in nodes_df.columns else 'stage'
desc_col = 'product_description' if 'product_description' in nodes_df.columns else 'products'

print(f"Kullanılan Sütunlar -> Stage: '{stage_col}', Desc: '{desc_col}'")

# Graph oluştur
G = nx.DiGraph()

# Node'ları eklerken "Hata çıkarsa boş geçme, Unknown yaz" mantığı
for _, row in nodes_df.iterrows():
    # .get() metodu hata vermez, bulamazsa None veya varsayılan değer döner
    G.add_node(
        row['id'],
        stage=row.get(stage_col, 'Unknown'),
        desc=row.get(desc_col, 'Unknown')
    )

# Edge'leri ekle
for _, row in edges_df.iterrows():
    G.add_edge(row['source'], row['target'])

# Part (Kaynak) node'larını tespit et
part_nodes = [n for n, attr in G.nodes(data=True) if attr.get('stage') == 'Part']

print("-" * 30)
print(f"Graph Başarıyla Kuruldu!")
print(f"Toplam Node: {len(G.nodes())}")
print(f"Toplam Edge: {len(G.edges())}")
print(f"Tespit Edilen 'Part' (Kaynak) Sayısı: {len(part_nodes)}")
print("Hazırlık Tamam. 2. Hücreye geçebilirsin.")

In [ ]:
from tqdm import tqdm

# --- ADIM 2: DISRUPTION SIMULATION (HESAPLAMA) ---
print("2. Simülasyon Başlıyor...")

disruption_data = []
all_nodes = list(G.nodes())

# İlerleme çubuğu ile takip et
for node_to_remove in tqdm(all_nodes, desc="Simulating Failures"):

    # 1. Sanal bir kopyada node'u sil
    H = G.copy()
    H.remove_node(node_to_remove)

    # 2. Hangi Part'lar hayatta kaldı?
    # Eğer silinen node bir 'Part' ise onu kaynak listesinden düş
    active_parts = [p for p in part_nodes if p != node_to_remove]

    # 3. Reachability (Erişilebilirlik) Analizi
    reachable_nodes = set()
    for part in active_parts:
        if part in H: # Hata önleyici kontrol
            reachable_nodes.add(part)
            reachable_nodes.update(nx.descendants(H, part))

    # 4. Kimler Aç Kaldı? (Starved)
    # Graph'ta kalan ama reachable olmayanlar
    remaining_nodes = set(H.nodes())
    starved_nodes = remaining_nodes - reachable_nodes

    # 5. Veriyi Kaydet (Sadece etkilenenleri 1 yap)
    row_result = {n: 1 for n in starved_nodes}
    row_result['DISRUPTED_NODE'] = node_to_remove

    disruption_data.append(row_result)

# DataFrame'e çevir
print("\nMatris Tablosu Oluşturuluyor (Biraz zaman alabilir)...")
disruption_matrix = pd.DataFrame(disruption_data).fillna(0)
disruption_matrix.set_index('DISRUPTED_NODE', inplace=True)

# Skorları Hesapla
vulnerability_scores = disruption_matrix.sum(axis=0).sort_values(ascending=False)
criticality_scores = disruption_matrix.sum(axis=1).sort_values(ascending=False)

print("Hesaplama Bitti. Skorlar hafızaya alındı. 3. Hücreye geçebilirsin.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- USER SETTINGS (DINAMIK AYARLAR) ---
NUM_NODES_TO_SHOW = 20  # Lollipop grafikte kaç node gösterilsin?
JITTER_STRENGTH = 0.8   # Scatter plot dağıtma şiddeti (Üst üste binmeyi engeller)

# --- DATA PREPARATION ---
# Ortak node'ları al
common_nodes = vulnerability_scores.index.intersection(criticality_scores.index)
x = criticality_scores.loc[common_nodes].values
y = vulnerability_scores.loc[common_nodes].values
labels = common_nodes

# --- VISUALIZATION 1: JITTERED SCATTER PLOT (GLOBAL RISK MAP) ---
plt.figure(figsize=(14, 10))

# Jittering: Add random noise to separate overlapping points
# (Note: This is purely for visual separation, actual data remains same)
x_jittered = x + np.random.normal(0, JITTER_STRENGTH, size=len(x))
y_jittered = y + np.random.normal(0, JITTER_STRENGTH, size=len(y))

# Plot with Jitter
sc = plt.scatter(x_jittered, y_jittered, c=y, cmap='coolwarm',
                 alpha=0.6, s=80, edgecolors='grey', linewidth=0.5)

# Quadrant Lines (Mean values)
mean_crit = np.mean(x)
mean_vuln = np.mean(y)
plt.axvline(x=mean_crit, color='gray', linestyle='--', alpha=0.5)
plt.axhline(y=mean_vuln, color='gray', linestyle='--', alpha=0.5)

# Zone Labels
plt.text(max(x)*0.95, max(y)*0.98, 'FATAL ZONE\n(High Crit + High Vuln)',
         color='#d62728', ha='right', va='top', fontweight='bold', fontsize=12)
plt.text(min(x), max(y)*0.98, 'VICTIMS\n(High Vuln)',
         color='#ff7f0e', ha='left', va='top', fontweight='bold', fontsize=12)
plt.text(max(x)*0.95, min(y), 'POWER PLAYERS\n(High Crit)',
         color='#2ca02c', ha='right', va='bottom', fontweight='bold', fontsize=12)

# Annotation: Label the absolute most risky nodes (Top 5 by combined risk)
risk_factor = x * y
top_risky_indices = np.argsort(risk_factor)[-5:]
for i in top_risky_indices:
    plt.annotate(
        f"{labels[i]}",
        (x_jittered[i], y_jittered[i]),
        xytext=(8, 8), textcoords='offset points',
        fontsize=9, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.8)
    )

plt.xlabel('CRITICALITY SCORE (Impact on others)', fontsize=11, fontweight='bold')
plt.ylabel('VULNERABILITY SCORE (Susceptibility to failure)', fontsize=11, fontweight='bold')
plt.title(f'Supply Chain Risk Quadrant (Jittered View)\nNote: Points slightly displaced to reveal overlaps', fontsize=14)
plt.colorbar(sc, label='Vulnerability Intensity')
plt.grid(True, alpha=0.2)
plt.show()

# --- VISUALIZATION 2: LOLLIPOP CHART (RANKING THE VULNERABLE) ---
# This chart solves the "Equal Score" visualization problem perfectly.

# Prepare Top N Data
top_vuln_data = vulnerability_scores.head(NUM_NODES_TO_SHOW)
top_vuln_indices = top_vuln_data.index
top_vuln_values = top_vuln_data.values

plt.figure(figsize=(12, len(top_vuln_data) * 0.4 + 2)) # Dynamic height

# Create Lollipop Chart
plt.hlines(y=range(len(top_vuln_data)), xmin=0, xmax=top_vuln_values, color='skyblue', alpha=0.8)
plt.plot(top_vuln_values, range(len(top_vuln_data)), "o", markersize=10, color='#d62728', alpha=0.9)

# Add Labels next to the dots
for i, (score, node_id) in enumerate(zip(top_vuln_values, top_vuln_indices)):
    # Get details for the label
    node_desc = G.nodes[node_id].get('desc', 'N/A')
    node_stage = G.nodes[node_id].get('stage', 'N/A')
    # Shorten description if too long
    if len(str(node_desc)) > 40: node_desc = str(node_desc)[:37] + "..."

    label_text = f"  Score: {int(score)} | {node_stage} | {node_desc}"
    plt.text(score, i, label_text, va='center', fontsize=9, color='#333333')

# Formatting
plt.yticks(range(len(top_vuln_data)), top_vuln_indices, fontsize=10, fontweight='bold')
plt.xlabel('Vulnerability Score (Number of Disruption Scenarios)', fontsize=11)
plt.title(f'Top {NUM_NODES_TO_SHOW} Most Vulnerable Nodes (Detailed Ranking)', fontsize=14)
plt.gca().invert_yaxis() # Highest score at top
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.xlim(0, max(top_vuln_values) * 1.4) # Make room for text labels
plt.tight_layout()
plt.show()

print(f"\n--- ANALYSIS SUMMARY ---")
print(f"1. Scatter Plot now uses 'Jittering'. If you see a cloud of dots, it means many nodes have similar risk profiles.")
print(f"2. Lollipop Chart lists the top {NUM_NODES_TO_SHOW} nodes explicitly.")
print(f"3. Equal scores in the Lollipop Chart confirm 'Structural Equivalence' (Shared Bottlenecks).")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

# --- 1. AYARLAR (BURAYI DEĞİŞTİREREK YÖNETEBİLİRSİN) ---
TOP_N_VULN = 10          # Kaç tane kırılgan node gösterelim? (Az tutmak netlik sağlar)
NODE_SIZE = 2500        # Node boyutu (İçine yazı sığması için büyük yaptım)
FONT_SIZE = 9           # Yazı boyutu

# --- 2. VERİ HAZIRLIĞI ---
# En kırılganları seç
top_vuln_nodes = vulnerability_scores.head(TOP_N_VULN).index.tolist()

# Sadece bu node'ları ve onların 1. derece bağlantılarını alalım
# (Çok geriye gidersek yine karışır, sadece doğrudan ilişkileri görelim)
nodes_to_draw = set(top_vuln_nodes)
for node in top_vuln_nodes:
    nodes_to_draw.update(G.predecessors(node))
    nodes_to_draw.update(G.successors(node))

H = G.subgraph(nodes_to_draw)

# --- 3. POZİSYONLAMA (GRID LAYOUT) ---
pos = {}
stage_map = {'Part': 0, 'Manuf': 1, 'Dist': 2, 'Retail': 3, 'Unknown': 4}
stage_colors = ['#e3f2fd', '#fff3e0', '#e8f5e9', '#f3e5f5', '#fafafa'] # Arka plan renkleri
stage_names = ["Parts\n(Supply)", "Manufacturing\n(Make)", "Distribution\n(Move)", "Retail\n(Sell)", "Unknown"]

# Node'ları stage'lerine göre grupla
nodes_by_stage = {0: [], 1: [], 2: [], 3: [], 4: []}
for node in H.nodes():
    stage = H.nodes[node].get('stage', 'Unknown')
    idx = stage_map.get(stage, 4)
    nodes_by_stage[idx].append(node)

# Koordinatları ata (Her sütunda eşit aralıklarla diz)
for idx, nodes in nodes_by_stage.items():
    nodes.sort(key=lambda x: vulnerability_scores.get(x, 0), reverse=True) # Skora göre sırala
    gap = 1.0 / (len(nodes) + 1)
    for i, node in enumerate(nodes):
        pos[node] = (idx, 1 - (i + 1) * gap)

# --- 4. ÇİZİM ---
plt.figure(figsize=(18, 10))
ax = plt.gca()

# A) Arka Plan Şeritlerini Çiz (Stages)
for i in range(4):
    plt.axvspan(i - 0.4, i + 0.4, color=stage_colors[i], alpha=0.6, zorder=0)
    plt.text(i, 1.02, stage_names[i], ha='center', fontsize=12, fontweight='bold', color='#444')

# B) Edge'leri Çiz
# Vulnerable olanlara giden okları Kırmızı yapalım, diğerleri Gri
edges = H.edges()
edge_colors = []
for u, v in edges:
    if v in top_vuln_nodes:
        edge_colors.append('#ff9999') # Hedef kırılgan ise açık kırmızı
    else:
        edge_colors.append('#bdbdbd') # Değilse gri

nx.draw_networkx_edges(H, pos, edge_color=edge_colors, width=1.5, arrowstyle='-|>', arrowsize=15, connectionstyle="arc3,rad=0.1")

# C) Node'ları Çiz
# Vulnerable olanlar MAVİ, Diğerleri BEYAZ
node_colors = ['#2196f3' if n in top_vuln_nodes else 'white' for n in H.nodes()]
edge_colors_node = ['#0d47a1' if n in top_vuln_nodes else '#9e9e9e' for n in H.nodes()]

nx.draw_networkx_nodes(H, pos, node_color=node_colors, node_size=NODE_SIZE,
                       edgecolors=edge_colors_node, linewidths=2)

# D) Etiketler (ID + SCORE)
labels = {}
for node in H.nodes():
    score = int(vulnerability_scores.get(node, 0))
    # Eğer Vulnerable listesindeyse skoru göster, değilse sadece ID
    if node in top_vuln_nodes:
        labels[node] = f"{node}\nScore:\n{score}"
    else:
        labels[node] = node

# Vulnerable olanların yazı rengi BEYAZ, diğerleri SİYAH
text_colors = {n: 'white' if n in top_vuln_nodes else 'black' for n in H.nodes()}
for node, (x, y) in pos.items():
    plt.text(x, y, labels[node], ha='center', va='center', fontsize=FONT_SIZE,
             fontweight='bold', color=text_colors[node])

plt.title(f"TOP {TOP_N_VULN} VULNERABLE NODES & CONNECTIVITY\n(Nodes in Blue are the most starved)", fontsize=16, pad=20)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# --- 1. AYARLAR ---
TOP_N_CRIT = 4   # En Kritik kaç node? (Azaltıldı: Netlik için)
TOP_N_VULN = 4   # En Kırılgan kaç node? (Azaltıldı: Netlik için)
NODE_SIZE = 2800

# --- 2. VERİ FİLTRELEME ---
crit_nodes = criticality_scores.head(TOP_N_CRIT).index.tolist()
vuln_nodes = vulnerability_scores.head(TOP_N_VULN).index.tolist()
fatal_nodes = set(crit_nodes).intersection(vuln_nodes)

# Sadece bu ana aktörleri ve onları birbirine bağlayan yolu alalım
nodes_to_draw = set(crit_nodes) | set(vuln_nodes)

# Ara bağlantıları bul (Shortest paths mantığıyla)
# Bu kısım grafiği çok temizleyecek: Sadece suçlu ve kurban arasındaki yolu göstereceğiz
path_nodes = set()
for source in crit_nodes:
    for target in vuln_nodes:
        if source != target and nx.has_path(G, source, target):
            # En kısa yoldaki tüm node'ları ekle
            try:
                paths = nx.all_shortest_paths(G, source, target)
                for path in paths:
                    path_nodes.update(path)
            except:
                pass

# Eğer hiç yol yoksa, sadece komşularını ekle
if not path_nodes:
    for n in nodes_to_draw:
        path_nodes.update(G.successors(n))
        path_nodes.update(G.predecessors(n))

nodes_to_draw.update(path_nodes)
H = G.subgraph(nodes_to_draw)

# --- 3. POZİSYONLAMA ---
pos = {}
nodes_by_stage = {0: [], 1: [], 2: [], 3: [], 4: []}
for node in H.nodes():
    stage = H.nodes[node].get('stage', 'Unknown')
    idx = stage_map.get(stage, 4)
    nodes_by_stage[idx].append(node)

for idx, nodes in nodes_by_stage.items():
    # Kritikleri yukarı, Kırılganları aşağı itmek yerine dengeli dağıtalım
    nodes.sort(key=lambda x: (x in crit_nodes, x in vuln_nodes), reverse=True)
    gap = 1.0 / (len(nodes) + 1)
    for i, node in enumerate(nodes):
        pos[node] = (idx, 1 - (i + 1) * gap)

# --- 4. ÇİZİM ---
plt.figure(figsize=(18, 10))

# Arka Plan Şeritleri
for i in range(4):
    plt.axvspan(i - 0.45, i + 0.45, color=stage_colors[i], alpha=0.4)
    plt.text(i, 1.02, stage_names[i], ha='center', fontsize=12, fontweight='bold', color='gray')

# Edge'ler (Critical -> Vulnerable akışını vurgula)
nx.draw_networkx_edges(H, pos, edge_color='gray', alpha=0.5, arrowstyle='-|>', arrowsize=15, connectionstyle="arc3,rad=0.1")

# Node Renkleri
node_colors = []
for n in H.nodes():
    if n in fatal_nodes: node_colors.append('purple') # Hem Kritik Hem Kırılgan
    elif n in crit_nodes: node_colors.append('#d32f2f') # Kritik (Kırmızı)
    elif n in vuln_nodes: node_colors.append('#1976d2') # Kırılgan (Mavi)
    else: node_colors.append('white') # Aradaki node (Beyaz)

# Node Çizimi
nx.draw_networkx_nodes(H, pos, node_color=node_colors, node_size=NODE_SIZE, edgecolors='#333', linewidths=2)

# Etiketler (ID + İlgili Skor)
labels = {}
text_colors = {}

for node in H.nodes():
    # Metin Rengi
    if node in crit_nodes or node in vuln_nodes:
        text_colors[node] = 'white'
    else:
        text_colors[node] = 'black'

    # Metin İçeriği
    if node in fatal_nodes:
        labels[node] = f"{node}\nFATAL\nC:{int(criticality_scores[node])}\nV:{int(vulnerability_scores[node])}"
    elif node in crit_nodes:
        labels[node] = f"{node}\nCRITICAL\nScore:{int(criticality_scores[node])}"
    elif node in vuln_nodes:
        labels[node] = f"{node}\nVULNERABLE\nScore:{int(vulnerability_scores[node])}"
    else:
        labels[node] = node

for node, (x, y) in pos.items():
    plt.text(x, y, labels[node], ha='center', va='center', fontsize=8, fontweight='bold', color=text_colors[node])

plt.title(f"THE KILL CHAIN: Interaction Map\n(Red: Source of Risk | Blue: Victim of Risk)", fontsize=16, pad=20)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# --- AYARLAR ---
# Her iki listede de ilk % kaçlık dilime gireni "Tehlikeli" sayalım?
# Örneğin ilk 30 node içinde kesişim arayalım.
SEARCH_DEPTH = 30

# --- 1. KESİŞİM ANALİZİ ---
top_crit_list = criticality_scores.head(SEARCH_DEPTH).index.tolist()
top_vuln_list = vulnerability_scores.head(SEARCH_DEPTH).index.tolist()

# Kesişim Kümesini Bul (Python Set Intersection)
fatal_nodes_set = set(top_crit_list).intersection(top_vuln_list)

# --- 2. DETAYLI TABLO OLUŞTURMA ---
fatal_data = []

if fatal_nodes_set:
    print(f"⚠️ UYARI: Hem Kritik hem Kırılgan {len(fatal_nodes_set)} adet 'FATAL NODE' tespit edildi!\n")

    for node in fatal_nodes_set:
        c_score = criticality_scores[node]
        v_score = vulnerability_scores[node]

        # Birleşik Risk Skoru (İkisinin çarpımı etkiyi daha iyi gösterir)
        composite_score = c_score * v_score

        # Node Bilgilerini Çek
        stage = G.nodes[node].get('stage', 'Unknown')
        desc = G.nodes[node].get('desc', 'Unknown')

        fatal_data.append({
            'Node ID': node,
            'Stage': stage,
            'Product': desc,
            'Criticality (Etki)': int(c_score),
            'Vulnerability (Kırılganlık)': int(v_score),
            'COMPOSITE RISK (Çarpım)': int(composite_score)
        })

    # DataFrame'e çevir ve Çarpım skoruna göre sırala
    fatal_df = pd.DataFrame(fatal_data).sort_values(by='COMPOSITE RISK (Çarpım)', ascending=False)

    # Tabloyu Göster
    # (Colab'de dataframe'i direkt yazdırmak güzel formatlı tablo verir)
    display(fatal_df)

    # CSV Olarak Kaydet (Rapor için)
    fatal_df.to_csv('FATAL_NODES_REPORT.csv', index=False)
    print("\n-> Detaylı liste 'FATAL_NODES_REPORT.csv' olarak kaydedildi.")

else:
    print("✅ İYİ HABER: Belirlenen derinlikte (Top " + str(SEARCH_DEPTH) + ") hem kritik hem kırılgan olan ortak bir node bulunamadı.")
    print("Bu, zincirdeki riskin dağıtık olduğunu gösterir (Tek bir merkezde toplanmamış).")

# --- 3. İLİŞKİ KATSAYISI (KORELASYON) ---
# Genel olarak Critical olanlar Vulnerable mı? (Tüm ağ için)
correlation = pd.Series(criticality_scores).corr(pd.Series(vulnerability_scores))
print(f"\nGenel Ağ Korelasyonu: {correlation:.4f}")
if correlation > 0.5:
    print("(Yorum: Yüksek pozitif ilişki. Büyük oyuncular aynı zamanda en riskli olanlar.)")
elif correlation < 0:
    print("(Yorum: Negatif ilişki. Kritik oyuncular genelde sağlam (robust) durumda.)")

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# --- ADIM 1: VERİLERİ YÜKLEME ---
print("Veriler yükleniyor...")
nodes_df = pd.read_csv('Pure-Nodes-Info.csv')
edges_df = pd.read_csv('PURE-Edges.csv')

# Graph oluşturma
G = nx.DiGraph()

# Node ekleme (Ürün bilgilerini 'desc' olarak kaydediyoruz)
for _, row in nodes_df.iterrows():
    G.add_node(row['id'], desc=row['products'], location=row.get('location', 'Unknown'))

# Edge ekleme
for _, row in edges_df.iterrows():
    G.add_edge(row['source'], row['target'])

# OTOMATİK ETİKETLEME: Giriş derecesi 0 olanlar 'Source' (Part) kabul edilir
source_nodes = [n for n, d in G.in_degree() if d == 0]
all_nodes = list(G.nodes())

print(f"Graf oluşturuldu. Toplam Node: {len(all_nodes)}, Tespit Edilen Kaynak (Part): {len(source_nodes)}")

# --- ADIM 2: HIZLI VULNERABILITY SİMÜLASYONU ---
# Her node'u tek tek devre dışı bırakıp etkilenenleri bulacağız
vulnerability_counts = {node: 0 for node in all_nodes}
criticality_counts = {node: 0 for node in all_nodes}

print("Analiz yapılıyor (Kademeli Etki Simülasyonu)...")

for node_to_remove in tqdm(all_nodes, desc="Simulating Failures"):
    # Bu node bozulduğunda etkilenebilecek alt düğümler (descendants)
    descendants = list(nx.descendants(G, node_to_remove))

    if not descendants:
        continue

    # Sanal olarak node'u ve bağlantılarını çıkarıyoruz
    # Verimlilik için sadece alt ağaca odaklanıyoruz
    remaining_nodes = set(G.nodes()) - {node_to_remove}
    sub_graph = G.subgraph(remaining_nodes)

    active_sources = [s for s in source_nodes if s != node_to_remove]

    # Hangi descendant artık hiçbir kaynağa ulaşamıyor?
    for victim in descendants:
        # Herhangi bir kaynağa yol var mı kontrol et
        has_access = False
        for s in active_sources:
            if nx.has_path(sub_graph, s, victim):
                has_access = True
                break

        if not has_access:
            vulnerability_counts[victim] += 1
            criticality_counts[node_to_remove] += 1

# --- ADIM 3: SONUÇLARI BİRLEŞTİRME VE RAPORLAMA ---
results = pd.DataFrame({
    'Node_ID': all_nodes,
    'Vulnerability_Score': [vulnerability_counts[n] for n in all_nodes],
    'Criticality_Score': [criticality_counts[n] for n in all_nodes]
})

# Ürün açıklamalarını geri ekle
results = results.merge(nodes_df[['id', 'products']], left_on='Node_ID', right_on='id').drop('id', axis=1)

# Normalizasyon ve Fatal Score (Risk Yükseltici)
# Hem kırılgan hem kritik olanları tespit eder
results['Fatal_Score'] = (results['Vulnerability_Score'] * results['Criticality_Score'])

# En Kırılgan 10 Node
print("\n" + "="*50)
print("EN KIRILGAN (VULNERABLE) 10 DÜĞÜM")
print("="*50)
top_vulnerable = results.sort_values(by='Vulnerability_Score', ascending=False).head(10)
print(top_vulnerable[['Node_ID', 'Vulnerability_Score', 'products']])

# --- ADIM 4: GÖRSELLEŞTİRME ---
plt.figure(figsize=(12, 6))
top_15_v = results.sort_values(by='Vulnerability_Score', ascending=False).head(15)
sns.barplot(data=top_15_v, x='Vulnerability_Score', y='Node_ID', palette='viridis')
plt.title('Tedarik Kesintisinden En Çok Etkilenen Düğümler (Top 15)')
plt.grid(axis='x', alpha=0.3)
plt.show()